# **深層生成モデル　第1回演習**

この演習では，**ベルヌーイ分布**を用いてMNISTデータセットをモデリングしていきます．また，人工データを用いて**多次元ガウス分布**のモデリングを学びます．

## 目次

1. [ベルヌーイ分布によるMNIST画像モデリング](#scrollTo=p-KulWxYNbXo)
2. [多次元ガウス分布による人工データモデリング](#scrollTo=FOeOCv32a9HN)

In [ ]:
# 必要なライブラリのインポート

import numpy as np
import matplotlib.pyplot as plt
import random
import copy
from keras.datasets import mnist
import seaborn as sns
import math
from scipy import stats
%matplotlib inline

In [ ]:
# ヘルパー関数の定義

def get_label_idxs(labels: list, t_mnist: np.ndarray) -> np.ndarray:
    # 長さ60000のt_mnistについて，値が指定したラベルのうちのいずれかであるかのboolean arrayを取ってくる
    label_bool = np.any([t_mnist==label for label in labels], axis=0)
    # Trueである要素のインデックスを得る
    label_idxs = np.where(label_bool)[0]

    return label_idxs


def transforms(data_all: np.ndarray, flatten=True, binarize=True) -> np.ndarray:
    # 範囲を0~255から0~1にし，平坦化したあと，閾値0.5で0,1のバイナリにする
    data_all = data_all.astype(np.float64) / 255
    if flatten:
        data_all = data_all.reshape((data_all.shape[0], -1))
    if binarize:
        data_all = (data_all > 0.5).astype(np.uint8)

    return data_all

## 1.ベルヌーイ分布によるMNIST画像モデリング


本セクションでは，MNIST画像の1種類のラベルのみを，単純な(多次元)ベルヌーイ分布でモデリングしてみます．

そのために，ここではラベルが1のサンプルのみを600枚取ってきます．

In [ ]:
NUM_SAMPLES = 600

(x_mnist, t_mnist), _ = mnist.load_data()
print(f"x_mnist: {x_mnist.shape}, t_mnist: {t_mnist.shape}")

labels = [1]

label_idxs = get_label_idxs(labels, t_mnist)

data_all = x_mnist[label_idxs][:NUM_SAMPLES]
print(f"data_all: {data_all.shape}")

最初のサンプルをプロットしてみます．

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(data_all[0], cmap='gray', annot=True, fmt='d', annot_kws={"fontsize":8}, square=True)

多次元ベルヌーイ分布によるモデリングを行うため，ピクセル値の範囲を0-255から0-1にし，{0,1}のバイナリにしてから，データを平坦化します．

In [ ]:
binary_data = transforms(data_all)
print(f"binary_data: {binary_data.shape}, set {np.unique(binary_data)}")

あるピクセルについて，値の分布をプロットしてみます．

In [ ]:
TARGET_PIXEL = 157

binary_pixel = binary_data[:, TARGET_PIXEL] # ( 600, )

print('0の回数: ', len(binary_pixel) - int(sum(binary_pixel)))
print('1の回数: ', int(sum(binary_pixel)))

plt.hist(binary_pixel)
plt.xlabel("Binarized pixel values")
plt.ylabel("Sample count")
plt.show()

各ピクセルそれぞれに個別のパラメータを持ったベルヌーイ分布が存在すると考えるので，ここでは一つのピクセルについてのみ記述します(つまり$D=784$のインデックスを表す$d$を省略)．

観測データ集合を$X=\left\{x_{i}\right\}_{i=1}^{N}$とします($N=600$)．$x_{i}$は$i$枚目の画像のあるピクセルの値を，閾値0.5で0, 1に2値化したものです．

生成モデルの分布としてパラメータが$\mu$のベルヌーイ分布を考え，モデルを設計します．

$$
p_{\mu}({\bf x})=\prod_{i=1}^{N} \mu^{x_{i}}(1-\mu)^{1-x_{i}}
$$

$\mu$の推定量$\hat{\mu}$を最尤推定で求めます．

$$
\hat{\mu}=\underset{\mu}{\operatorname{argmax}} \sum_{i=1}^{N}\left[x_{i} \log \mu+\left(1-x_{i}\right) \log (1-\mu)\right]
$$

対数尤度関数を$\mu$について偏微分してゼロとおくことで，最尤推定量は次のように求まります．

$$
\hat{\mu}=\frac{1}{N}\sum_{i=1}^{N} x_{i}
$$

In [ ]:
# 最尤推定
mu_hats = # WRITE ME
print(f'mu_hats: {mu_hats.shape}') # ( 784, )

# TARGET_PIXELのmu_hatをパラメータとするベルヌーイ分布からデータを生成する
sampled_data = stats.bernoulli.rvs(p=mu_hats[TARGET_PIXEL], size=NUM_SAMPLES)
print('0の回数: ', len(sampled_data) - sum(sampled_data))
print('1の回数: ', sum(sampled_data))

plt.hist(sampled_data)
plt.xlabel("Binarized pixel values")
plt.ylabel("Sample count")
plt.show()

ベルヌーイ分布の$\hat{\mu}$を直接可視化してみます．ある程度1を生成するようなパラメータ集合が得られていることがわかります．

In [ ]:
plt.imshow(mu_hats.reshape(28, 28), cmap="gray")

次に，ラベルが0のサンプルを混ぜて，同じ多次元ベルヌーイ分布でモデリングを行ってみます．

In [ ]:
NUM_SAMPLES = 1200

labels = [0, 1]

label_idxs = get_label_idxs(labels, t_mnist)
data_all = x_mnist[label_idxs][:NUM_SAMPLES]
print(f"data_all: {data_all.shape}")

binary_data = transforms(data_all)
print(f"binary_data: {binary_data.shape}, set {np.unique(binary_data)}")

# 最尤推定
mu_hats = np.mean(binary_data, axis=0) # ( 784, )
print(f'mu_hats: {mu_hats.shape}')

plt.imshow(mu_hats.reshape(28, 28), cmap="gray")

0とも1とも言えない画像を生成するようなパラメータが得られてしまいました．

データのソースに明らかに複数の確率分布が考えられる場合は，潜在変数を考慮することが有用です．次回の講義では潜在変数を含めた混合分布とその学習について扱います．

## 2.多次元ガウス分布による人工データモデリング

本セクションで用いるデータは，人工的に生成した2次元ガウス分布によるデータです．

In [ ]:
# 真の分布（データ分布）として単一のガウス分布を定義
#（以下のパラメータは真の分布のパラメータなので，推定（学習）時にはわからないことに注意）
_mu = np.array([0, 0])  # 平均
_sigma = np.array([[1.0, 0.5], [0.5, 1.0]])  # 共分散行列

NUM_DATA = 3000

# ガウス分布からサンプリング（訓練データ）
data_all = np.random.multivariate_normal(_mu, _sigma, NUM_DATA)

print(f"data_all: {data_all.shape}")

In [ ]:
#データ点の可視化
NUM_SAMPLES = 300

x = np.arange(-5, 5, 0.05) # x軸
y = np.arange(-5, 5, 0.05) # y軸

fig, ax = plt.subplots(figsize=(8.0, 6.0))
data_plot = plt.scatter(data_all.T[0, :NUM_SAMPLES], data_all.T[1, :NUM_SAMPLES], s=10, c='g')
ax.set_aspect('equal','box')
plt.grid(True)
plt.show()

生成モデルの分布として,パラメータが $\boldsymbol{\mu}$（平均ベクトル）と $\boldsymbol{\Sigma}$（共分散行列）の **多次元ガウス分布** を考え，モデルを設計します．

$$
p_{\boldsymbol{\mu}, \boldsymbol{\Sigma}}(\mathbf{x}) = \frac{1}{(2\pi)^{d/2} |\boldsymbol{\Sigma}|^{1/2}} \exp\left( -\frac{1}{2} (\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu}) \right)
$$

$\boldsymbol{\mu}$ の推定量 $\hat{\boldsymbol{\mu}}$ と $\boldsymbol{\Sigma}$ の推定量 $\hat{\boldsymbol{\Sigma}}$ を 最尤推定で求めます．

対数尤度関数は次のようになります．

$$
\log p_{\boldsymbol{\mu}, \boldsymbol{\Sigma}}(\mathbf{X}) = -\frac{N d}{2} \log(2\pi) - \frac{N}{2} \log |\boldsymbol{\Sigma}| - \frac{1}{2} \sum_{i=1}^{N} (\mathbf{x}_i - \boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}_i - \boldsymbol{\mu})
$$

この式を $\boldsymbol{\mu}$ と $\boldsymbol{\Sigma}$ について偏微分し，ゼロとおくことで最尤推定量が求まります．

- 平均ベクトルの最尤推定量:

$$
\hat{\boldsymbol{\mu}} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{x}_i
$$

- 共分散行列の最尤推定量:

$$
\hat{\boldsymbol{\Sigma}} = \frac{1}{N} \sum_{i=1}^{N} (\mathbf{x}_i - \hat{\boldsymbol{\mu}})(\mathbf{x}_i - \hat{\boldsymbol{\mu}})^T
$$

この結果より，最尤推定では データの平均と共分散 を計算することで，ガウス分布のパラメータを推定できることが分かります．

In [ ]:
# データの平均と共分散を求める関数
def estimate_gaussian_params(data_all):
    mu = np.mean(data_all, axis=0)  # 平均
    sigma = np.cov(data_all, rowvar=False)  # 共分散行列
    return mu, sigma

# 単一のガウス分布を推定
mu_hats, sigma_hats = estimate_gaussian_params(data_all)

推定したパラメータに基づくガウス分布を可視化してみます．

In [ ]:
# 共分散行列の逆行列と行列式
sigma_hats_inv = np.linalg.inv(sigma_hats)
sigma_hats_det = np.linalg.det(sigma_hats)

# メッシュグリッドの作成
x = np.arange(-5, 5, 0.05)  # x軸
y = np.arange(-5, 5, 0.05)  # y軸
X, Y = np.meshgrid(x, y)  # (200, 200)

# ガウス密度関数
def gaussian_density(x, mu, sigma_inv, sigma_det):
    diff = x - mu
    exponent = -0.5 * diff.T @ sigma_inv @ diff
    return np.exp(exponent) / (2 * np.pi * np.sqrt(sigma_det))

# 密度の計算
Z = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(Y.shape[1]):
        Z[i, j] = gaussian_density(np.array([X[i, j], Y[i, j]]), mu_hats, sigma_hats_inv, sigma_hats_det)

# データ点の可視化
fig, ax = plt.subplots(figsize=(8.0, 6.0))
data_plot = plt.scatter(data_all[:NUM_SAMPLES, 0], data_all[:NUM_SAMPLES, 1], s=10, c='g')
contour = ax.contour(X, Y, Z, levels=[0.01 * i for i in range(11)], cmap='cool')
plt.colorbar(contour)
ax.set_aspect('equal', 'box')
plt.grid(True)
plt.show()

また推定したガウス分布は生成モデルなので，元のデータと類似した新しいデータ点を生成することもできます．

In [ ]:
# 新しいデータの生成
NUM_GENERATED = 100
new_data = # WRITE ME

In [ ]:
# データ点の可視化
fig, ax = plt.subplots(figsize=(8.0, 6.0))
data_plot = plt.scatter(data_all[:NUM_SAMPLES, 0], data_all[:NUM_SAMPLES, 1], s=10, c='g', label='Original Data')
generated_plot = plt.scatter(new_data[:, 0], new_data[:, 1], s=10, c='r', label='Generated Data')
contour = ax.contour(X, Y, Z, levels=np.linspace(Z.min(), Z.max(), 10), cmap='cool')
plt.colorbar(contour)
ax.set_aspect('equal', 'box')
plt.legend()
plt.grid(True)
plt.show()